# Assignment: When Chat Systems Break - A Realistic Sharding Simulation


---
## 📅 Day 8 (8 April): Hash-Based Sharding — Better but Not Perfect

In [ ]:
import random


class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content


class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

    def count(self):
        return len(self.messages)


class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def send_message(self, message):
        # will be implemented by subclasses
        pass

    def print_stats(self):
        total = sum(s.count() for s in self.shards)
        print(f"{'Shard':<10} {'Messages':<15} {'% of Total':<10}")
        print('-' * 35)
        for shard in self.shards:
            pct = round((shard.count() / total * 100), 1) if total > 0 else 0
            print(f"Shard {shard.id:<5} {shard.count():<15} {pct}%")
        print(f"{'Total':<10} {total}")


In [ ]:
# =============================================
# DAY 8: Hash-Based Sharding
# Use MD5 hash to distribute more evenly
# Decision: what do we hash? user_id? channel_id? message_id?
# =============================================

import hashlib

class HashShardManager(ShardManager):

    def get_shard(self, key):
        # MD5 hash gives a big number, we use % to pick a shard
        h = int(hashlib.md5(str(key).encode()).hexdigest(), 16)
        return self.shards[h % len(self.shards)]

    def send_message(self, message):
        # DECISION: We hash channel_id
        # Why? Because we want messages of the same channel together,
        # and hash spreads channels more evenly than simple modulo.
        shard = self.get_shard(message.channel_id)
        shard.store(message)


# --- Same viral scenario as Day 7 ---
print("=== Hash-Based Sharding Simulation ===")
print("Same scenario: Channel 3 (Cricket Final) is viral (80% traffic)")
print("Key choice: hashing channel_id")
print()

hash_manager = HashShardManager(num_shards=3)

for i in range(8000):
    msg = Message(user_id=random.randint(1, 50000), channel_id=3, content="INDIA WINS!")
    hash_manager.send_message(msg)

for i in range(2000):
    ch_id = random.choice([1, 2, 4, 5, 6, 7, 8, 9, 10])
    msg = Message(user_id=random.randint(1, 50000), channel_id=ch_id, content="hello")
    hash_manager.send_message(msg)

print("--- Messages per Shard ---")
hash_manager.print_stats()
print()
print("--- Observation ---")
print("Hash of channel_id=3 still maps to ONE shard.")
print("So hash-based sharding does NOT solve the hotspot problem")
print("when ONE channel dominates traffic.")
print()
print("--- Why Hash? What's better about it? ---")
print("If traffic is SPREAD across many channels, hash gives better distribution.")
print("Example: channels 1-50 each with equal traffic → hash spreads them evenly.")
print()
print("--- Key Choice Explanation ---")
print("Hash user_id   → same problem as Day 6 (one user = one shard)")
print("Hash channel_id → channel messages stay together (good for reads)")
print("Hash message_id → very even spread, but queries are harder (Day 10 problem)")
print("We chose channel_id as it keeps related messages in one place.")
print()
print("--- The Big Limitation: Shard Count Changes ---")
print("If we had 3 shards and now add 1 more (total 4):")
print("  Old: hash(channel_3) % 3 = some shard")
print("  New: hash(channel_3) % 4 = DIFFERENT shard")
print("  All existing data is now in the WRONG shard! System breaks.")